데이터준비

In [1]:
import zipfile, os

zip_path = "phishing URL dataset.zip"  # 다운로드한 파일 이름 맞게 변경
extract_path = "data"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ 압축 해제 완료:", os.listdir(extract_path))

✅ 압축 해제 완료: ['Phishing URL dataset']


In [2]:
import os

print(os.listdir("data/Phishing URL dataset"))


['Phishing URLs.csv', 'URL dataset.csv']


In [3]:
## 데이터셋 불러와서 형태확인
import pandas as pd

df1 = pd.read_csv(r"C:/Users/monbe/Phising/data/Phishing URL dataset/Phishing URLs.csv")
df2 = pd.read_csv(r"C:/Users/monbe/Phising/data/Phishing URL dataset/URL dataset.csv")

print("df1 컬럼:", df1.columns.tolist())## 피싱
print("df2 컬럼:", df2.columns.tolist())## 정상

df1.head(), df2.head()


df1 컬럼: ['url', 'Type']
df2 컬럼: ['url', 'type']


(                                                 url      Type
 0  https://docs.google.com/presentation/d/e/2PACX...  Phishing
 1    https://btttelecommunniccatiion.weeblysite.com/  Phishing
 2                        https://kq0hgp.webwave.dev/  Phishing
 3  https://brittishtele1bt-69836.getresponsesite....  Phishing
 4         https://bt-internet-105056.weeblysite.com/  Phishing,
                          url        type
 0     https://www.google.com  legitimate
 1    https://www.youtube.com  legitimate
 2   https://www.facebook.com  legitimate
 3      https://www.baidu.com  legitimate
 4  https://www.wikipedia.org  legitimate)

In [4]:
## 두 csv를 하나로 합치기 -> 학습용 통합 데이터셋 만들어짐
import pandas as pd

# 파일 읽기
df_phishing = pd.read_csv(r"C:/Users/monbe/Phising/data/Phishing URL dataset/Phishing URLs.csv")
df_legit = pd.read_csv(r"C:/Users/monbe/Phising/data/Phishing URL dataset/URL dataset.csv")

# 컬럼 이름統一
df_phishing.columns = ['url', 'label']
df_legit.columns = ['url', 'label']

# 라벨 값 변환
df_phishing['label'] = 1        # 피싱 (1)
df_legit['label'] = 0           # 정상 (0)

# 합치기
df = pd.concat([df_phishing, df_legit], axis=0).reset_index(drop=True)

print("총 데이터 수:", len(df))
df.head()


총 데이터 수: 504983


,url,label
0,https://docs.google.com/presentation/d/e/2PACX...,1
1,https://btttelecommunniccatiion.weeblysite.com/,1
2,https://kq0hgp.webwave.dev/,1
3,https://brittishtele1bt-69836.getresponsesite....,1
4,https://bt-internet-105056.weeblysite.com/,1


In [5]:
# 데이터 정제 (중복/결측 제거)
df = df.drop_duplicates(subset=['url']) # 중복행 제거
df = df.dropna(subset=['url', 'label']) # 결측갑 제거
df = df.reset_index(drop=True) # DataFrame의 인덱스 재설정

print("정제 완료:", df.shape)
df.head()

정제 완료: (504933, 2)


,url,label
0,https://docs.google.com/presentation/d/e/2PACX...,1
1,https://btttelecommunniccatiion.weeblysite.com/,1
2,https://kq0hgp.webwave.dev/,1
3,https://brittishtele1bt-69836.getresponsesite....,1
4,https://bt-internet-105056.weeblysite.com/,1


In [6]:
#정상 URL 500개 생성 (아직 합치지 않음!)
import random

base_urls = [
    "https://www.naver.com", "https://naver.com",
    "https://www.daum.net", "https://daum.net",
    "https://www.kakao.com", "https://kakao.com",
    "https://www.nate.com", "https://nate.com",
    "https://google.co.kr", "https://www.google.co.kr",

    "https://www.coupang.com", "https://coupang.com",
    "https://www.gmarket.co.kr", "https://www.11st.co.kr",

    "https://www.kbfg.com", "https://obank.kbstar.com",
    "https://www.shinhan.com", "https://bank.shinhan.com",
    "https://www.wooribank.com", "https://www.ibk.co.kr",
    "https://www.nhbank.com", "https://www.citibank.co.kr",

    "https://www.gov.kr", "https://www.hometax.go.kr",
    "https://www.nts.go.kr", "https://www.korea.kr"
]

paths = [
    "/", "/main", "/home", "/login", "/search", "/news",
    "/support", "/help", "/customer", "/about", "/event",
    "/item", "/products"
]

extended = []
for url in base_urls:
    extended.append(url)
    for _ in range(8):
        extended.append(url + random.choice(paths))

df_normal_korea = pd.DataFrame({
    "url": extended[:500],
    "label": 0
})

df_normal_korea.head(), df_normal_korea.shape
df_normal_korea.to_csv("normal_korea_500.csv", index=False, encoding="utf-8-sig")


In [7]:
#원본 df다운샘플링
# 해결: 다운샘플링
from sklearn.utils import resample

# 피싱 / 정상 분리
df_phish = df[df['label'] == 1]
df_legit = df[df['label'] == 0]

# 정상 데이터 다운샘플링
df_legit_down = resample(df_legit,
                         replace=False,
                         n_samples=len(df_phish)*1,  # 피싱과 동일 개수로 하기
                         random_state=42)

# 합치기
df_balanced = pd.concat([df_phish, df_legit_down]).sample(frac=1, random_state=42).reset_index(drop=True)

print("샘플링 후 라벨 분포:")
print(df_balanced['label'].value_counts())

df_balanced.to_csv("downsampled_dataset.csv", index=False)


샘플링 후 라벨 분포:
0    54805
1    54805
Name: label, dtype: int64


In [8]:
#다운샘플링된 df_balanced + 한국 URL 500개 합치기
df_v2 = pd.concat([df_balanced, df_normal_korea], ignore_index=True)
df_v2 = df_v2.sample(frac=1, random_state=42).reset_index(drop=True)

df_v2["label"].value_counts()
df_v2.shape


(109844, 2)

모델학습

In [2]:
import pandas as pd

df = pd.read_csv(
    r"C:\Users\monbe\Phising\data\Phishing URL dataset\downsampled_dataset.csv"
)

df.head()
df.shape


(109610, 2)

In [3]:
#Feature Engineering (URL 기반 숫자 특징 생성)

import re
from urllib.parse import urlparse

def extract_features(url):
    parsed = urlparse(url)
    domain = parsed.netloc

    url_length = len(url)
    domain_length = len(domain)
    num_dots = url.count(".")
    num_digits = sum(c.isdigit() for c in url)
    num_hyphens = url.count("-")
    num_special_chars = len(re.findall(r"[^a-zA-Z0-9]", url))
    has_https = 1 if url.startswith("https") else 0
    has_ip = 1 if re.match(r"^https?://\d+\.\d+\.\d+\.\d+", url) else 0

    suspicious_words = ["login", "verify", "update", "secure", "account", "reset","signin"]
    contains_suspicious = 1 if any(w in url.lower() for w in suspicious_words) else 0

    return pd.Series([
        url_length, domain_length, num_dots, num_digits,
        num_hyphens, num_special_chars, has_https, has_ip,
        contains_suspicious
    ])

df[
    [
        "url_length","domain_length","num_dots","num_digits",
        "num_hyphens","num_special_chars","has_https",
        "has_ip","contains_suspicious"
    ]
] = df["url"].apply(extract_features)

df.head()


,url,label,url_length,domain_length,num_dots,num_digits,num_hyphens,num_special_chars,has_https,has_ip,contains_suspicious
0,https://www.pittsburghsymphony.org/pghsymph.ns...,0,85,26,3,22,0,9,1,0,0
1,http://onceaunicorn.com/wp-includes/pomo/file/...,0,58,16,2,0,1,11,0,0,0
2,https://www.lifescript.com/doctor-directory/ho...,0,106,18,3,0,8,17,1,0,0
3,https://storage.googleapis.com/trustusaa/signi...,0,52,22,3,0,0,8,1,0,1
4,https://btinternet-104787.weeblysite.com/,1,41,32,2,6,1,7,1,0,0


In [4]:
#TF-IDF 벡터화
#v2버전 - Feature Engineering + TF-IDF 결합 모델
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF: URL 문자 패턴을 벡터화
vectorizer = TfidfVectorizer(
    analyzer='char_wb',   # URL은 단어보다 문자 패턴이 중요
    ngram_range=(3,5)     # 3~5글자 조각 분석
)

X_tfidf = vectorizer.fit_transform(df['url'])
X_tfidf.shape



(109610, 2667914)

In [6]:
#숫자 Feature와 TF-IDF 결합

from scipy.sparse import hstack

# Feature Engineering으로 만든 컬럼들
feature_cols = [
    "url_length","domain_length","num_dots","num_digits",
    "num_hyphens","num_special_chars","has_https",
    "has_ip","contains_suspicious"
]

# 숫자 특징만 따로 추출 (numpy array)
X_features = df[feature_cols].values  

# TF-IDF 벡터 + 숫자 Feature 결합
X = hstack([X_tfidf, X_features])

# 레이블
y = df["label"]


In [7]:
# TRAIN/TEST Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

X_train.shape, X_test.shape


((87688, 2667923), (21922, 2667923))

In [8]:
#Logistic Regression 학습

from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=2000,  # 복잡한 피처 + tfidf를 위해 반복 수 증가
    n_jobs=-1       # CPU 전체 사용
)

model.fit(X_train, y_train)
model


LogisticRegression(max_iter=2000, n_jobs=-1)

성능평가

In [10]:
from sklearn.metrics import classification_report

pred = model.predict(X_test)
print(classification_report(y_test, pred))


              precision    recall  f1-score   support

           0       0.96      0.98      0.97     10961
           1       0.98      0.96      0.97     10961

    accuracy                           0.97     21922
   macro avg       0.97      0.97      0.97     21922
weighted avg       0.97      0.97      0.97     21922



v2 모델 저장

In [11]:
import joblib

# TF-IDF 벡터 변환기 저장
joblib.dump(vectorizer, "vectorizer.pkl")

# 학습된 최종 V2 모델 저장
joblib.dump(model, "phish_model.pkl")

print("모델 저장 완료!")


모델 저장 완료!


모델 불러오기(다시학습할필요없음)

In [12]:
import joblib

vectorizer = joblib.load("vectorizer.pkl")
model = joblib.load("phish_model.pkl")
